# HOMEWORK 13

Done by: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/ComputerVision-PBY_HW13

In this homework you are going to inspect the GTSRB (German Traffic Sign Recognition Benchmark) dataset. The dataset contains images of various classes of traffic signs used in Germany (and the whole EU). The objective of this homework is to go through the steps described below and to implement the necessary code.

At the end, as usual, there will be a couple of questions for you to answer. In addition, the last section of this homework is optional and, if you chose to do it, you'll earn extra point :-)

In [7]:
# Silent installation or update

import sys
from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError, version

# Clean cache
!"{sys.executable}" -m pip cache purge -q

# Force updating
package_update = [
    "pip",
    "scikit-learn",
    "pandas",
    "kagglehub",
    "kagglesdk",
]

for package_name in package_update:
    !"{sys.executable}" -m pip install -U "{package_name}" -q

# Install missing packages
package_array = [
    "jinja2",
    "ipywidgets",
    "nbformat",
    "numpy",
    "matplotlib",
    "opencv-contrib-python",
]


def is_package_satisfied(req_str: str) -> bool:
    try:
        req = Requirement(req_str)
        installed_ver = version(req.name)
        if req.specifier:
            return installed_ver in req.specifier
        return True
    except PackageNotFoundError:
        return False


for package_name in package_array:
    if not is_package_satisfied(package_name):
        !"{sys.executable}" -m pip install "{package_name}" -q


In [ ]:
# Synchronization with remote source

import shutil
from pathlib import Path

# Input images
hm_version = 13

# Solution
git_project_url = f"https://github.com/BogdanPinchuk/ComputerVision-PBY_HW{hm_version}.git"
main_file_name = f"Bohdan_Pinchuk_CV_HW{hm_version}.ipynb"

# upload all files
current_path = !pwd
current_path = current_path[0]
parent_path = !dirname "$current_path"
parent_path = parent_path[0]
temp_path = f"{parent_path}/temp"

# Clone images
!rm -rf "$temp_path"
!git clone "$git_project_url" "$temp_path"

source = Path(temp_path)
destination = Path(current_path)
exclude = {main_file_name, ".git", ".idea"}

for item in source.iterdir():
    if item.name in exclude:
        continue

    target = destination / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Clean temp folder
!rm -rf "$temp_path"

# Note. To upload big files to GitHub, execute the next command in the terminal
# 1. brew install git-lfs
# 2. git lfs install
# 3.1. git lfs track "*.safetensors"
# 3.2. git lfs track "*.bin"

### Step 0

Go to the GTSRB dataset official site ([link](https://benchmark.ini.rub.de/gtsrb_dataset.html)) to learn more about the dataset.

### Step 1

Download the dataset ([link](https://www.kaggle.com/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign)) and unzip it.

In [ ]:
# Downloading data

import pandas as pd
import apps.main as mn

# Input data
update_db = False
db_file_name = f"resources/data/store_hw{hm_version}.db"
ds_handle = "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
ds_file_name = "Train.csv"

# Solution
pd.options.display.float_format = "{:g}".format  # type: ignore
gtsrb_dataset, dataset_root = mn.download_and_extract_from_kagglehub(
    ds_handle,
    ds_file_name,
    db_file_name,
    update_db=update_db,
)

# Print result
print(f"Dataset root directory: {dataset_root}")
display(gtsrb_dataset)


### Step 2

For this homework, you will be working with the training set. Check out the `Train.csv`, open it and see what it contains. Load the dataset and plot random samples.

In [ ]:
# Load the training labels
root = '...'  # Path to the dataset location, e.g., '/data/janko/dataset/GTSRB'
data = pd.read_csv(os.path.join(root, 'Train.csv'))

# Number of training samples (amount of samples in data)
num_samples = ...  # TODO

# Show random data samples
for ii in range(15):
    # Get random index
    idx = np.random.randint(0, num_samples)
    # Load image
    img = cv2.imread(os.path.join(root, data.iloc[idx]['Path']))
    # Convert image to RGB
    img = ...  # TODO
    # Show image
    plt.subplot(3, 5, ii + 1), plt.imshow(img), plt.title(data.iloc[idx]['ClassId'])

### Step 3

Inspect the dataset by computing and plotting the per-class histogram.

In [ ]:
# Extract class identifiers
# Hint: Check the csv 
ids = ...  # TODO

Compute the per class histogram. You can use any approach you want (e.g. `numpy`). It's also worth looking at the `Counter` function from the `collections` module ([link](https://docs.python.org/3/library/collections.html#collections.Counter)) ;-)

In [ ]:
from collections import Counter

hist = Counter(ids)

plt.bar(hist.keys(), hist.values()), plt.grid(True)
plt.xlabel('Traffic Sign ID'), plt.ylabel('Counts')

### Questions

Please answer the following questions:
* Do you consider the dataset to be balanced? If so, why? If not, why?
* Are there any classes that are (significantly) over-represented or under-represented?
* What would be the accuracy of a "classifier" that always predicts the most frequent class? What does it tell you about using accuracy as the only metric for this dataset?
* The images are `uint8` with values in [0, 255] and they have different sizes. What preprocessing would you apply before feeding them to a neural network (and why)?

### Optional

Perform a further analysis on the dataset and draw some conclusion from it.

Hint 1: Unlike MNIST or CIFAR10, this dataset contains images with various spatial resolutions. Is there anything we can tell about the resolution distribution?
Hint 2: What about the brightness distribution? Are there classes there are significantly more bright than others?